In [1]:
!pip install torch faiss-gpu

  Using cached torch-2.14.1-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (37 kB)
  Using cached faiss_gpu-1.15.1-cp310-abi3-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (8.1 kB)
  Using cached filelock-4.0.9-py3-none-any.whl.metadata (2.0 kB)
  Using cached setuptools-84.0.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.7-py3-none-any.whl.metadata (6.7 kB)
  Using cached fsspec-2026.9.0-py3-none-any.whl.metadata (10 kB)
  Using cached cuda_toolkit-13.0.3.0-py2.py3-none-any.whl.metadata (17 kB)
  Using cached cuda_bindings-13.4.3-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (2.4 kB)
  Using cached nvidia_cudnn_cu13-9.24.0.43-py3-none-manylinux_2_27_x86_64.whl.metadata (1.9 kB)
  Using cached nvidia_cusparselt_cu13-0.8.1-py3-none-manylinux2014_x86_64.whl.metadata (12 kB)
  Using cached nvidia_nccl_cu13-2.30.7-py3-none-manylinux_2_18_x86_64.whl.metadata (2.1 kB)
  Using cache

In [2]:
!pip install colbert-ai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 25.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 23.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 18.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.6/111.6 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 387.7/387.7 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.4/103.4 kB 58.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.5/228.5 kB 40.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 33.8 MB/s eta 0:00:00m eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 49.5 MB/s eta 0:00:00m eta 0:00:01

In [32]:
!pip uninstall -y transformers
!pip install transformers==4.48.3

Found existing installation: transformers 4.48.3
Uninstalling transformers-4.48.3:
  Successfully uninstalled transformers-4.48.3
  Using cached transformers-4.48.3-py3-none-any.whl.metadata (44 kB)
Using cached transformers-4.48.3-py3-none-any.whl (9.7 MB)


# ColBERT Performance on SciFact

## Downloading Dataset

In [17]:
import ir_datasets
from pathlib import Path
import os

1
0
NVIDIA H200 NVL


In [25]:
dataset = ir_datasets.load('beir/scifact')
docs_count = dataset.docs_count()

dataset_name = 'scifact'
outpath = Path('/ml_data/aryan') / dataset_name
i_to_id = dict()
os.makedirs(outpath, exist_ok=True)
with open(outpath / "collections.tsv" , "w", encoding='utf-8') as f:
    i = 0
    for doc in dataset.docs_iter():
        id_ = doc[0] #id
        text = doc[1] #text
        title = doc[2] # title
        content = f"{title} {text}".strip()
        i_to_id[i] = id_
        f.write(f"{i}\t{content}\n")
        i += 1

[INFO] Opening /home/aryan/.ir_datasets/beir/scifact/docs.pklz4/bin with direct file access


## Indexing Collection

In [7]:
from colbert.infra import Run, RunConfig, ColBERTConfig
from colbert import Indexer

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

/home/aryan/ColBERT/.venv/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


In [8]:
def main():
  with Run().context(RunConfig(nranks=1, experiment="scifact")):

    config = ColBERTConfig(
        nbits=2,
        root="/ml_data/aryan",
    )

    indexer = Indexer(checkpoint="jinaai/jina-colbert-v1-en", config=config)
    indexer.index(name="scifact.nbits=2", collection="/ml_data/aryan/scifact/collections.tsv")

In [9]:
if __name__ == "__main__":
  main()

artifact.metadata: 0.00B [00:00, ?B/s]



[Oct 02, 10:24:17] #> Creating directory /home/aryan/ColBERT/finetuning/experiments/scifact/indexes/scifact.nbits=2 


#> Starting...


/home/aryan/ColBERT/.venv/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


nranks = 1 	 num_gpus = 1 	 device=0
{
    "query_token_id": "[unused0]",
    "doc_token_id": "[unused1]",
    "query_token": "[Q]",
    "doc_token": "[D]",
    "ncells": null,
    "centroid_score_threshold": null,
    "ndocs": null,
    "load_index_with_mmap": false,
    "index_path": null,
    "index_bsize": 64,
    "nbits": 2,
    "kmeans_niters": 4,
    "resume": false,
    "pool_factor": 1,
    "clustering_mode": "hierarchical",
    "protected_tokens": 0,
    "similarity": "cosine",
    "bsize": 64,
    "accumsteps": 1,
    "lr": 1e-5,
    "maxsteps": 400000,
    "save_every": null,
    "warmup": 20000,
    "warmup_bert": null,
    "relu": false,
    "nway": 64,
    "use_ib_negatives": true,
    "reranker": false,
    "distillation_alpha": 1.0,
    "ignore_scores": false,
    "model_name": null,
    "query_maxlen": 32,
    "attend_to_mask_tokens": false,
    "interaction": "colbert",
    "dim": 128,
    "doc_maxlen": 160,
    "mask_punctuation": true,
    "checkpoint": "jinaai\/ji

A new version of the following files was downloaded from https://huggingface.co/jinaai/jina-bert-implementation:
- configuration_bert.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/jinaai/jina-bert-implementation:
- modeling_bert.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
/home/aryan/ColBERT/.venv/lib/python3.12/site-packages/colbert/utils/amp.py:12: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler()


...

[Oct 02, 10:24:46] [0] 		 avg_doclen_est = 141.1859893798828 	 len(local_sample) = 5,183
[Oct 02, 10:24:46] [0] 		 Creating 8,192 partitions.
[Oct 02, 10:24:46] [0] 		 *Estimated* 731,766 embeddings.
[Oct 02, 10:24:46] [0] 		 #> Saving the indexing plan to /home/aryan/ColBERT/finetuning/experiments/scifact/indexes/scifact.nbits=2/plan.json ..
Clustering 695179 points in 128D to 8192 clusters, redo 1 times, 4 iterations
  Preprocessing in 0.05 s
[Oct 02, 10:24:47] Loading decompress_residuals_cpp extension (set COLBERT_LOAD_TORCH_EXTENSION_VERBOSE=True for more info)...
[Oct 02, 10:25:50] Loading packbits_cpp extension (set COLBERT_LOAD_TORCH_EXTENSION_VERBOSE=True for more info)...
[0.036, 0.034, 0.036, 0.038, 0.038, 0.035, 0.037, 0.035, 0.035, 0.036, 0.038, 0.036, 0.038, 0.034, 0.035, 0.038, 0.038, 0.039, 0.036, 0.041, 0.037, 0.036, 0.036, 0.037, 0.038, 0.035, 0.036, 0.037, 0.036, 0.036, 0.038, 0.038, 0.037, 0.035, 0.038, 0.033, 0.035, 0.037, 0.037, 0.039, 0.035, 0.038, 0.038, 0.036, 

0it [00:00, ?it/s]/home/aryan/ColBERT/.venv/lib/python3.12/site-packages/colbert/utils/amp.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  return torch.cuda.amp.autocast() if self.activated else NullContextManager()


[Oct 02, 10:26:55] [0] 		 #> Saving chunk 0: 	 5,183 passages and 731,767 embeddings. From #0 onward.
[Oct 02, 10:26:55] [0] 		 #> Checking all files were saved...
[Oct 02, 10:26:55] [0] 		 Found all files!
[Oct 02, 10:26:55] [0] 		 #> Building IVF...
[Oct 02, 10:26:55] [0] 		 #> Loading codes...
[Oct 02, 10:26:55] [0] 		 Sorting codes...
[Oct 02, 10:26:55] [0] 		 Getting unique codes...
[Oct 02, 10:26:55] #> Optimizing IVF to store map from centroids to list of pids..
[Oct 02, 10:26:55] #> Building the emb2pid mapping..
[Oct 02, 10:26:55] len(emb2pid) = 731767
[Oct 02, 10:26:55] #> Saved optimized IVF to /home/aryan/ColBERT/finetuning/experiments/scifact/indexes/scifact.nbits=2/ivf.pid.pt
[Oct 02, 10:26:55] [0] 		 #> Saving the indexing metadata to /home/aryan/ColBERT/finetuning/experiments/scifact/indexes/scifact.nbits=2/metadata.json ..



1it [00:02,  2.42s/it]
100%|██████████| 8192/8192 [00:00<00:00, 130112.08it/s]


#> Joined...


In [26]:
import json
with open("/ml_data/aryan/scifact/i-to-ids.json", "w", encoding='utf-8') as f:
    f.write(json.dumps(i_to_id, ensure_ascii=False))

In [27]:
dataset = ir_datasets.load('beir/scifact/test')

dataset_name = 'scifact'
outpath = Path('/ml_data/aryan') / dataset_name
i_to_qid = dict()
os.makedirs(outpath, exist_ok=True)
with open(outpath / "queries_test.tsv" , "w", encoding='utf-8') as f:
    i = 0
    for doc in dataset.queries_iter():
        qid_ = doc[0] #id
        query = doc[1] #text
        i_to_qid[i] = qid_
        f.write(f"{i}\t{query}\n")
        i += 1

In [28]:
import json
with open("/ml_data/aryan/scifact/i-to-qids.json", "w", encoding='utf-8') as f:
    f.write(json.dumps(i_to_qid, ensure_ascii=False))

In [17]:
from colbert.data import Queries
from colbert.infra import Run, RunConfig, ColBERTConfig
from colbert import Searcher

if __name__=='__main__':
    with Run().context(RunConfig(nranks=1, experiment="scifact")):

        config = ColBERTConfig(
            root="/ml_data/aryan",
        )
        searcher = Searcher(index="scifact.nbits=2", config=config, checkpoint="jinaai/jina-colbert-v1-en")
        queries = Queries("/ml_data/aryan/scifact/queries_test.tsv")
        ranking = searcher.search_all(queries, k=100)
        ranking.save("scifact.nbits=2.ranking.tsv")

[Oct 02, 10:28:40] #> Loading collection...
0M 
[Oct 02, 10:28:47] #> Loading codec...
[Oct 02, 10:28:47] #> Loading IVF...
[Oct 02, 10:28:47] #> Loading doclens...


/home/aryan/ColBERT/.venv/lib/python3.12/site-packages/colbert/utils/amp.py:12: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler()
100%|████████████| 1/1 [00:00<00:00, 1837.99it/s]

[Oct 02, 10:28:47] #> Loading codes and residuals...



100%|██████████████| 1/1 [00:00<00:00, 25.15it/s]

[Oct 02, 10:28:47] #> Loading the queries from /ml_data/aryan/scifact/queries_test.tsv ...
[Oct 02, 10:28:47] #> Got 300 queries. All QIDs are unique.




/home/aryan/ColBERT/.venv/lib/python3.12/site-packages/colbert/utils/amp.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  return torch.cuda.amp.autocast() if self.activated else NullContextManager()
300it [00:01, 209.16it/s]



[Oct 02, 10:28:49] #> Creating directory /home/aryan/ColBERT/finetuning/experiments/scifact/none/2026-10/02/10.23.53 


[Oct 02, 10:28:49] #> Saved ranking of 300 queries and 30000 lines to /home/aryan/ColBERT/finetuning/experiments/scifact/none/2026-10/02/10.23.53/scifact.nbits=2.ranking.tsv


In [20]:
import pandas as pd
df = pd.read_csv("experiments/scifact/none/2026-10/02/10.23.53/scifact.nbits=2.ranking.tsv", sep='\t', names=['qid', 'docid', 'rank', 'score'])

df.head()

,qid,docid,rank,score
0,0,2723,1,14.929688
1,0,2686,2,13.960938
2,0,5062,3,13.500000
3,0,5147,4,13.468750
4,0,2635,5,13.429688


In [21]:
from collections import defaultdict

preds = defaultdict(list)
for i, row in df.iterrows():
    qid, docid, _, _ = row
    actual_qid = i_to_qid[qid]
    actual_docid = i_to_id[docid]

    preds[actual_qid].append(actual_docid)

In [23]:
!pip install scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 5.4 MB/s eta 0:00:00m eta 0:00:010:01:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 306.1/306.1 kB 2.9 MB/s eta 0:00:002.9 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 474.0/474.0 kB 2.5 MB/s eta 0:00:002.3 MB/s eta 0:00:01


In [8]:
import numpy as np
from sklearn.metrics import ndcg_score

In [9]:
def compute_metrics(hits: dict, rel: dict, save_path: str = None):  
    ndcg_scores = []
    rr = []
    recalls = []
    ap_scores = []
    for qid, ret_doc in hits.items():
        rel_doc = rel[qid]

        # NDCG Score
        y_true = np.array([[1 if doc in rel_doc else 0 for doc in ret_doc[:10]]])
        y_score = np.array([[len(ret_doc) - i for i in range(len(ret_doc[:10]))]])

        ndcg = ndcg_score(y_true, y_score, k=10)
        ndcg_scores.append(ndcg)

        # Recall
        top100= set(ret_doc)
        count = 0
        for d in rel_doc:
            if d in top100:
                count += 1
        recall = count / len(rel_doc) if rel_doc else 0
        recalls.append(recall)

        # MRR
        rank = 0
        for i, d in enumerate(ret_doc[:10]):
            if d in rel_doc:
                rank = i + 1
                break

        if rank > 0:
            rr.append(1 / rank)
        else:
            rr.append(0)

        #MAP
        num_hits = 0
        precisions = []
        for i, d in enumerate(ret_doc[:10]):
            if d in rel_doc:
                num_hits += 1
                precisions.append(num_hits / (i + 1))

        ap = sum(precisions) / min(len(rel_doc), 10) if rel_doc else 0
        ap_scores.append(ap)

    avg_ndcg = np.mean(np.array(ndcg_scores))
    print(f"NDCG@10: {avg_ndcg:.4f}")

    mrr = np.mean(np.array(rr))
    print(f"MRR@10: {mrr:.4f}")

    recall = np.mean(np.array(recalls))
    print(f"Recall@100: {recall:.4f}")

    avg_ap = np.mean(np.array(ap_scores))
    print(f"MAP@10: {avg_ap:.4f}")

    return avg_ndcg, mrr, recall, avg_ap

In [41]:
dataset = ir_datasets.load('beir/scifact/test')

dataset_name = 'scifact'
outpath = Path('/ml_data/aryan') / dataset_name
qrels = defaultdict(list)
os.makedirs(outpath, exist_ok=True)
with open(outpath / "qrels_test.tsv" , "w", encoding='utf-8') as f:
    for doc in dataset.qrels_iter():
        qid_ = doc[0] #qid
        docid_ = doc[1] #docid
        qrels[qid_].append(docid_)
        f.write(f"{qid_}\t{docid_}\n")

In [27]:
compute_metrics(preds, qrels);

NDCG@10: 0.6589
MRR@10: 0.6226
Recall@100: 0.9057
MAP@10: 0.6077


## Fine-tuning on SciFact


In [8]:
import os

os.environ['CUDA_VISIBLE_DEVICES'] = '2'

import json
from collections import defaultdict
from pathlib import Path

import ir_datasets
from colbert import Trainer
from colbert.infra import Run, RunConfig, ColBERTConfig

/home/aryan/ColBERT/.venv/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


In [9]:
dataset = ir_datasets.load('beir/scifact/train')

dataset_name = 'scifact'
outpath = Path('/ml_data/aryan') / dataset_name
i_to_train_qid = dict()
train_qrels = defaultdict(list)
id_to_i = {docid: i for i, docid in i_to_id.items()}
os.makedirs(outpath, exist_ok=True)
with open(outpath / 'queries_train.tsv', 'w', encoding='utf-8') as f:
    for i, doc in enumerate(dataset.queries_iter()):
        qid_ = doc[0]
        query = doc[1]
        i_to_train_qid[i] = qid_
        f.write(f'{i}\t{query}\n')

with open(outpath / 'qrels_train.tsv', 'w', encoding='utf-8') as f:
    for doc in dataset.qrels_iter():
        qid_ = doc[0]
        docid_ = doc[1]
        train_qrels[qid_].append(id_to_i[docid_])
        f.write(f'{qid_}\t{docid_}\n')


In [10]:
from pyserini.index.lucene import LuceneIndexer

index_path = outpath / 'lucene_index'
indexer = LuceneIndexer(str(index_path))
with open(outpath / 'collections.tsv', 'r', encoding='utf-8') as f:
    for line in f:
        docid, content = line.rstrip('\n').split('\t', 1)
        indexer.add_doc_dict({'id': docid, 'contents': content})
indexer.close()


2026-10-02 11:02:08,747 INFO  [main] index.SimpleIndexer (SimpleIndexer.java:138) - Using DefaultEnglishAnalyzer
2026-10-02 11:02:08,748 INFO  [main] index.SimpleIndexer (SimpleIndexer.java:139) - Stemmer: porter
2026-10-02 11:02:08,748 INFO  [main] index.SimpleIndexer (SimpleIndexer.java:140) - Keep stopwords? false
2026-10-02 11:02:08,748 INFO  [main] index.SimpleIndexer (SimpleIndexer.java:141) - Stopwords file: null


In [12]:
os.environ["OPENAI_API_KEY"] = "dummy"
from pyserini.search.lucene import LuceneSearcher

searcher = LuceneSearcher(str(outpath / 'lucene_index'))
train_qid_to_i = {qid: i for i, qid in i_to_train_qid.items()}
with open(outpath / 'queries_train.tsv', 'r', encoding='utf-8') as f:
    train_queries = {int(line.split('\t', 1)[0]): line.rstrip('\n').split('\t', 1)[1] for line in f}

with open(outpath / 'triples_train.jsonl', 'w', encoding='utf-8') as f:
    for qid_, positives in train_qrels.items():
        qid = train_qid_to_i[qid_]
        hits = searcher.search(train_queries[qid], k=10)
        negative = next((int(hit.docid) for hit in hits if int(hit.docid) not in positives), None)
        if negative is not None:
            for positive in positives:
                f.write(json.dumps([qid, positive, negative]) + '\n')


In [18]:
def main():
  with Run().context(RunConfig(nranks=1, experiment='scifact-finetune')):

    config = ColBERTConfig(
        bsize=16,
        lr=3e-6,
        warmup=5,
        maxsteps=50,
        nway=2,
        use_ib_negatives=True,
        root='/ml_data/aryan',
    )

    trainer = Trainer(
        triples='/ml_data/aryan/scifact/triples_train.jsonl',
        queries='/ml_data/aryan/scifact/queries_train.tsv',
        collection='/ml_data/aryan/scifact/collections.tsv',
        config=config,
    )

    trainer.train(checkpoint='jinaai/jina-colbert-v1-en')

if __name__ == '__main__':
  main()


#> Starting...


/home/aryan/ColBERT/.venv/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


nranks = 1 	 num_gpus = 1 	 device=0
{
    "query_token_id": "[unused0]",
    "doc_token_id": "[unused1]",
    "query_token": "[Q]",
    "doc_token": "[D]",
    "ncells": null,
    "centroid_score_threshold": null,
    "ndocs": null,
    "load_index_with_mmap": false,
    "index_path": null,
    "index_bsize": 64,
    "nbits": 1,
    "kmeans_niters": 4,
    "resume": false,
    "pool_factor": 1,
    "clustering_mode": "hierarchical",
    "protected_tokens": 0,
    "similarity": "cosine",
    "bsize": 16,
    "accumsteps": 1,
    "lr": 3e-6,
    "maxsteps": 50,
    "save_every": null,
    "warmup": 5,
    "warmup_bert": null,
    "relu": false,
    "nway": 2,
    "use_ib_negatives": true,
    "reranker": false,
    "distillation_alpha": 1.0,
    "ignore_scores": false,
    "model_name": null,
    "query_maxlen": 32,
    "attend_to_mask_tokens": false,
    "interaction": "colbert",
    "dim": 128,
    "doc_maxlen": 220,
    "mask_punctuation": true,
    "checkpoint": "jinaai\/jina-colber

/home/aryan/ColBERT/.venv/lib/python3.12/site-packages/colbert/utils/amp.py:12: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler()
/home/aryan/ColBERT/.venv/lib/python3.12/site-packages/colbert/utils/amp.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  return torch.cuda.amp.autocast() if self.activated else NullContextManager()
/home/aryan/ColBERT/.venv/lib/python3.12/site-packages/colbert/utils/amp.py:35: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-ra


#> LR will use 5 warmup steps and linear decay over 50 steps.

#> QueryTokenizer.tensorize(batch_text[0], batch_background[0], bsize) ==
#> Input: 0-dimensional biomaterials lack inductive properties., 		 True, 		 None
#> Output IDs: torch.Size([32]), tensor([  101,     1,  1014,  1011,  8789, 16012,  8585, 14482,  2015,  3768,
        27427, 14194,  6024,  5144,  1012,   102,   103,   103,   103,   103,
          103,   103,   103,   103,   103,   103,   103,   103,   103,   103,
          103,   103], device='cuda:0')
#> Output Mask: torch.Size([32]), tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0], device='cuda:0')

				 1.6206409931182861 1.4870051145553589
#>>>    16.76 16.64 		|		 0.120000000000001
[Oct 02, 11:11:41] 0 3.1076459884643555
				 0.7827466726303101 1.3288464546203613
#>>>    19.51 17.41 		|		 2.1000000000000014
[Oct 02, 11:11:41] 1 3.106649935722351
				 1.8984328508377075 2.171354055404663
#>>>    18.3

## Evaluate Fine-tuned Checkpoint


In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"

import torch

print(torch.cuda.device_count())     
print(torch.cuda.current_device())    
print(torch.cuda.get_device_name(0)) 

1
0
NVIDIA H200 NVL


In [2]:
from pathlib import Path

In [4]:
from colbert import Indexer, Searcher
from colbert.data import Queries
from colbert.infra import Run, RunConfig, ColBERTConfig

CHECKPOINT_PATH = Path('/home/aryan/ColBERT/finetuning/experiments/scifact-finetune/none/2026-10/02/11.02.00/checkpoints/colbert')

assert CHECKPOINT_PATH.exists()

/home/aryan/ColBERT/.venv/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


In [29]:
with Run().context(RunConfig(nranks=1, experiment='scifact-finetune-eval')):
    index_config = ColBERTConfig(nbits=2, root='/ml_data/aryan')
    indexer = Indexer(checkpoint=str(CHECKPOINT_PATH), config=index_config)
    indexer.index(
        name='scifact.finetuned.nbits=2',
        collection='/ml_data/aryan/scifact/collections.tsv',
    )


AssertionError: /home/aryan/ColBERT/finetuning/experiments/scifact-finetune-eval/indexes/scifact.finetuned.nbits=2

In [5]:
import transformers
transformers.__version__

'4.48.3'

In [6]:
with Run().context(RunConfig(nranks=1, experiment='scifact-finetune-eval')):
    search_config = ColBERTConfig(root='/ml_data/aryan')
    searcher = Searcher(
        index='scifact.finetuned.nbits=2',
        checkpoint=str(CHECKPOINT_PATH),
        config=search_config,
    )
    test_queries = Queries('/ml_data/aryan/scifact/queries_test.tsv')
    finetuned_ranking = searcher.search_all(test_queries, k=100)
    finetuned_ranking_path = finetuned_ranking.save('scifact.finetuned.nbits=2.ranking.tsv')

[Oct 02, 11:24:29] #> Loading collection...
0M 
[Oct 02, 11:24:33] #> Loading codec...
[Oct 02, 11:24:33] Loading decompress_residuals_cpp extension (set COLBERT_LOAD_TORCH_EXTENSION_VERBOSE=True for more info)...


/home/aryan/ColBERT/.venv/lib/python3.12/site-packages/colbert/utils/amp.py:12: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler()


[Oct 02, 11:24:33] Loading packbits_cpp extension (set COLBERT_LOAD_TORCH_EXTENSION_VERBOSE=True for more info)...
[Oct 02, 11:24:33] #> Loading IVF...
[Oct 02, 11:24:33] #> Loading doclens...


100%|█| 1/1 [00:00<00:00, 1918.71i

[Oct 02, 11:24:33] #> Loading codes and residuals...



100%|█| 1/1 [00:00<00:00, 23.94it/

[Oct 02, 11:24:33] #> Loading the queries from /ml_data/aryan/scifact/queries_test.tsv ...
[Oct 02, 11:24:33] #> Got 300 queries. All QIDs are unique.




/home/aryan/ColBERT/.venv/lib/python3.12/site-packages/colbert/utils/amp.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  return torch.cuda.amp.autocast() if self.activated else NullContextManager()
300it [00:01, 212.73it/s]




[Oct 02, 11:24:35] #> Creating directory /home/aryan/ColBERT/finetuning/experiments/scifact-finetune-eval/none/2026-10/02/11.24.20 


[Oct 02, 11:24:35] #> Saved ranking of 300 queries and 30000 lines to /home/aryan/ColBERT/finetuning/experiments/scifact-finetune-eval/none/2026-10/02/11.24.20/scifact.finetuned.nbits=2.ranking.tsv


In [8]:
from collections import defaultdict

In [29]:
import json
with open("/ml_data/aryan/scifact/i-to-ids.json", "r", encoding='utf-8') as f:
    i_to_id = json.load(f)

with open("/ml_data/aryan/scifact/i-to-qids.json", "r", encoding='utf-8') as f:
    i_to_qid = json.load(f)


In [43]:
preds = defaultdict(list)
for qid, results in finetuned_ranking.items():
    actual_qid = i_to_qid[str(qid)]
    for docid, _, _ in results:
        preds[actual_qid].append(i_to_id[str(docid)])

compute_metrics(preds, qrels);

NDCG@10: 0.6631
MRR@10: 0.6250
Recall@100: 0.8990
MAP@10: 0.6117


# Changed Hyperparams

In [1]:
import os

os.environ['CUDA_VISIBLE_DEVICES'] = '2'

import json
from collections import defaultdict
from pathlib import Path

import ir_datasets
from colbert import Trainer
from colbert.infra import Run, RunConfig, ColBERTConfig

/home/aryan/ColBERT/.venv/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


In [2]:
def main():
  with Run().context(RunConfig(nranks=1, experiment='scifact-finetune-hparam-v2')):

    config = ColBERTConfig(
        bsize=16,
        lr=1e-5,
        warmup=18,
        maxsteps=174,
        nway=2,
        use_ib_negatives=True,
        root='/ml_data/aryan',
    )

    trainer = Trainer(
        triples='/ml_data/aryan/scifact/triples_train.jsonl',
        queries='/ml_data/aryan/scifact/queries_train.tsv',
        collection='/ml_data/aryan/scifact/collections.tsv',
        config=config,
    )

    trainer.train(checkpoint='jinaai/jina-colbert-v1-en')

if __name__ == '__main__':
  main()


#> Starting...


/home/aryan/ColBERT/.venv/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


nranks = 1 	 num_gpus = 1 	 device=0
{
    "query_token_id": "[unused0]",
    "doc_token_id": "[unused1]",
    "query_token": "[Q]",
    "doc_token": "[D]",
    "ncells": null,
    "centroid_score_threshold": null,
    "ndocs": null,
    "load_index_with_mmap": false,
    "index_path": null,
    "index_bsize": 64,
    "nbits": 1,
    "kmeans_niters": 4,
    "resume": false,
    "pool_factor": 1,
    "clustering_mode": "hierarchical",
    "protected_tokens": 0,
    "similarity": "cosine",
    "bsize": 16,
    "accumsteps": 1,
    "lr": 1e-5,
    "maxsteps": 174,
    "save_every": null,
    "warmup": 18,
    "warmup_bert": null,
    "relu": false,
    "nway": 2,
    "use_ib_negatives": true,
    "reranker": false,
    "distillation_alpha": 1.0,
    "ignore_scores": false,
    "model_name": null,
    "query_maxlen": 32,
    "attend_to_mask_tokens": false,
    "interaction": "colbert",
    "dim": 128,
    "doc_maxlen": 220,
    "mask_punctuation": true,
    "checkpoint": "jinaai\/jina-colb

/home/aryan/ColBERT/.venv/lib/python3.12/site-packages/colbert/utils/amp.py:12: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler()
/home/aryan/ColBERT/.venv/lib/python3.12/site-packages/colbert/utils/amp.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  return torch.cuda.amp.autocast() if self.activated else NullContextManager()
/home/aryan/ColBERT/.venv/lib/python3.12/site-packages/colbert/utils/amp.py:35: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-ra


#> LR will use 18 warmup steps and linear decay over 174 steps.

#> QueryTokenizer.tensorize(batch_text[0], batch_background[0], bsize) ==
#> Input: 0-dimensional biomaterials lack inductive properties., 		 True, 		 None
#> Output IDs: torch.Size([32]), tensor([  101,     1,  1014,  1011,  8789, 16012,  8585, 14482,  2015,  3768,
        27427, 14194,  6024,  5144,  1012,   102,   103,   103,   103,   103,
          103,   103,   103,   103,   103,   103,   103,   103,   103,   103,
          103,   103], device='cuda:0')
#> Output Mask: torch.Size([32]), tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0], device='cuda:0')

				 1.6206409931182861 1.4870051145553589
#>>>    16.76 16.64 		|		 0.120000000000001
[Oct 03, 05:24:41] 0 3.1076459884643555
				 0.7827466726303101 1.3288464546203613
#>>>    19.51 17.41 		|		 2.1000000000000014
[Oct 03, 05:24:41] 1 3.106649935722351
				 1.8972430229187012 2.1707119941711426
#>>>    1

In [3]:
from colbert import Indexer, Searcher
from colbert.data import Queries
from colbert.infra import Run, RunConfig, ColBERTConfig

CHECKPOINT_PATH = Path('/home/aryan/ColBERT/finetuning/experiments/scifact-finetune-hparam-v2/none/2026-10/03/05.19.27/checkpoints/colbert')

assert CHECKPOINT_PATH.exists()

In [4]:
with Run().context(RunConfig(nranks=1, experiment='scifact-finetune-hparam-v2')):
    index_config = ColBERTConfig(nbits=2, root='/ml_data/aryan')
    indexer = Indexer(checkpoint=str(CHECKPOINT_PATH), config=index_config)
    indexer.index(
        name='scifact.finetuned.nbits=2',
        collection='/ml_data/aryan/scifact/collections.tsv',
    )




[Oct 03, 05:25:22] #> Creating directory /home/aryan/ColBERT/finetuning/experiments/scifact-finetune-hparam-v2/indexes/scifact.finetuned.nbits=2 


#> Starting...


/home/aryan/ColBERT/.venv/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


nranks = 1 	 num_gpus = 1 	 device=0
{
    "query_token_id": "[unused0]",
    "doc_token_id": "[unused1]",
    "query_token": "[Q]",
    "doc_token": "[D]",
    "ncells": null,
    "centroid_score_threshold": null,
    "ndocs": null,
    "load_index_with_mmap": false,
    "index_path": null,
    "index_bsize": 64,
    "nbits": 2,
    "kmeans_niters": 4,
    "resume": false,
    "pool_factor": 1,
    "clustering_mode": "hierarchical",
    "protected_tokens": 0,
    "similarity": "cosine",
    "bsize": 64,
    "accumsteps": 1,
    "lr": 1e-5,
    "maxsteps": 174,
    "save_every": null,
    "warmup": 18,
    "warmup_bert": null,
    "relu": false,
    "nway": 2,
    "use_ib_negatives": true,
    "reranker": false,
    "distillation_alpha": 1.0,
    "ignore_scores": false,
    "model_name": null,
    "query_maxlen": 32,
    "attend_to_mask_tokens": false,
    "interaction": "colbert",
    "dim": 128,
    "doc_maxlen": 160,
    "mask_punctuation": true,
    "checkpoint": "\/home\/aryan\/Co

/home/aryan/ColBERT/.venv/lib/python3.12/site-packages/colbert/utils/amp.py:12: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler()



[Oct 03, 05:25:29] [0] 		 # of sampled PIDs = 5183 	 sampled_pids[:3] = [3412, 83, 2446]
[Oct 03, 05:25:29] [0] 		 #> Encoding 5183 passages..


...

[Oct 03, 05:25:32] [0] 		 avg_doclen_est = 141.1859893798828 	 len(local_sample) = 5,183
[Oct 03, 05:25:32] [0] 		 Creating 8,192 partitions.
[Oct 03, 05:25:32] [0] 		 *Estimated* 731,766 embeddings.
[Oct 03, 05:25:32] [0] 		 #> Saving the indexing plan to /home/aryan/ColBERT/finetuning/experiments/scifact-finetune-hparam-v2/indexes/scifact.finetuned.nbits=2/plan.json ..
Clustering 695179 points in 128D to 8192 clusters, redo 1 times, 4 iterations
  Preprocessing in 0.05 s
[Oct 03, 05:25:33] Loading decompress_residuals_cpp extension (set COLBERT_LOAD_TORCH_EXTENSION_VERBOSE=True for more info)...
[Oct 03, 05:25:33] Loading packbits_cpp extension (set COLBERT_LOAD_TORCH_EXTENSION_VERBOSE=True for more info)...
[0.034, 0.033, 0.035, 0.036, 0.036, 0.033, 0.035, 0.034, 0.034, 0.034, 0.038, 0.034, 0.037, 0.033, 0.034, 0.036, 0.035, 0.038, 0.035, 0.04, 0.035, 0.034, 0.035, 0.034, 0.036, 0.033, 0.035, 0.035, 0.034, 0.035, 0.037, 0.036, 0.034, 0.034, 0.036, 0.032, 0.033, 0.036, 0.035, 0.037, 

0it [00:00, ?it/s]/home/aryan/ColBERT/.venv/lib/python3.12/site-packages/colbert/utils/amp.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  return torch.cuda.amp.autocast() if self.activated else NullContextManager()


[Oct 03, 05:25:36] [0] 		 #> Saving chunk 0: 	 5,183 passages and 731,767 embeddings. From #0 onward.
[Oct 03, 05:25:36] [0] 		 #> Checking all files were saved...
[Oct 03, 05:25:36] [0] 		 Found all files!
[Oct 03, 05:25:36] [0] 		 #> Building IVF...
[Oct 03, 05:25:36] [0] 		 #> Loading codes...
[Oct 03, 05:25:36] [0] 		 Sorting codes...
[Oct 03, 05:25:36] [0] 		 Getting unique codes...
[Oct 03, 05:25:36] #> Optimizing IVF to store map from centroids to list of pids..
[Oct 03, 05:25:36] #> Building the emb2pid mapping..
[Oct 03, 05:25:36] len(emb2pid) = 731767
[Oct 03, 05:25:36] #> Saved optimized IVF to /home/aryan/ColBERT/finetuning/experiments/scifact-finetune-hparam-v2/indexes/scifact.finetuned.nbits=2/ivf.pid.pt
[Oct 03, 05:25:36] [0] 		 #> Saving the indexing metadata to /home/aryan/ColBERT/finetuning/experiments/scifact-finetune-hparam-v2/indexes/scifact.finetuned.nbits=2/metadata.json ..


1it [00:02,  2.50s/it]
100%|██████████| 8192/8192 [00:00<00:00, 128905.41it/s]



#> Joined...


In [7]:
with Run().context(RunConfig(nranks=1, experiment='scifact-finetune-eval')):
    search_config = ColBERTConfig(root='/ml_data/aryan')
    searcher = Searcher(
        index='scifact.finetuned.nbits=2',
        checkpoint=str(CHECKPOINT_PATH),
        config=search_config,
    )
    test_queries = Queries('/ml_data/aryan/scifact/queries_test.tsv')
    finetuned_ranking = searcher.search_all(test_queries, k=100)
    finetuned_ranking_path = finetuned_ranking.save('scifact.finetuned.nbits=2.ranking.tsv')

[Oct 03, 05:26:41] #> Loading collection...
0M 
[Oct 03, 05:26:45] #> Loading codec...
[Oct 03, 05:26:45] Loading decompress_residuals_cpp extension (set COLBERT_LOAD_TORCH_EXTENSION_VERBOSE=True for more info)...


/home/aryan/ColBERT/.venv/lib/python3.12/site-packages/colbert/utils/amp.py:12: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler()


[Oct 03, 05:26:45] Loading packbits_cpp extension (set COLBERT_LOAD_TORCH_EXTENSION_VERBOSE=True for more info)...
[Oct 03, 05:26:45] #> Loading IVF...
[Oct 03, 05:26:45] #> Loading doclens...


100%|███████████████| 1/1 [00:00<00:00, 2145.42it/s]

[Oct 03, 05:26:45] #> Loading codes and residuals...



100%|█████████████████| 1/1 [00:00<00:00, 39.24it/s]

[Oct 03, 05:26:45] #> Loading the queries from /ml_data/aryan/scifact/queries_test.tsv ...
[Oct 03, 05:26:45] #> Got 300 queries. All QIDs are unique.




/home/aryan/ColBERT/.venv/lib/python3.12/site-packages/colbert/utils/amp.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  return torch.cuda.amp.autocast() if self.activated else NullContextManager()
300it [00:01, 225.61it/s]



[Oct 03, 05:26:47] #> Creating directory /home/aryan/ColBERT/finetuning/experiments/scifact-finetune-eval/none/2026-10/03/05.19.27 


[Oct 03, 05:26:47] #> Saved ranking of 300 queries and 30000 lines to /home/aryan/ColBERT/finetuning/experiments/scifact-finetune-eval/none/2026-10/03/05.19.27/scifact.finetuned.nbits=2.ranking.tsv


In [5]:
from collections import defaultdict

import json
with open("/ml_data/aryan/scifact/i-to-ids.json", "r", encoding='utf-8') as f:
    i_to_id = json.load(f)

with open("/ml_data/aryan/scifact/i-to-qids.json", "r", encoding='utf-8') as f:
    i_to_qid = json.load(f)


In [12]:
dataset = ir_datasets.load('beir/scifact/test')

dataset_name = 'scifact'
outpath = Path('/ml_data/aryan') / dataset_name
qrels = defaultdict(list)
os.makedirs(outpath, exist_ok=True)
with open(outpath / "qrels_test.tsv" , "w", encoding='utf-8') as f:
    for doc in dataset.qrels_iter():
        qid_ = doc[0] #qid
        docid_ = doc[1] #docid
        qrels[qid_].append(docid_)
        f.write(f"{qid_}\t{docid_}\n")

In [13]:
preds = defaultdict(list)
for qid, results in finetuned_ranking.items():
    actual_qid = i_to_qid[str(qid)]
    for docid, _, _ in results:
        preds[actual_qid].append(i_to_id[str(docid)])

compute_metrics(preds, qrels);

NDCG@10: 0.6700
MRR@10: 0.6296
Recall@100: 0.9063
MAP@10: 0.6162


# Longer Training

In [14]:
import os

os.environ['CUDA_VISIBLE_DEVICES'] = '2'

import json
from collections import defaultdict
from pathlib import Path

import ir_datasets
from colbert import Trainer
from colbert.infra import Run, RunConfig, ColBERTConfig

In [15]:
def main():
  with Run().context(RunConfig(nranks=1, experiment='scifact-finetune-longer')):

    config = ColBERTConfig(
        bsize=16,
        lr=7e-6,
        warmup=30,
        maxsteps=290,
        nway=2,
        use_ib_negatives=True,
        root='/ml_data/aryan',
    )

    trainer = Trainer(
        triples='/ml_data/aryan/scifact/triples_train.jsonl',
        queries='/ml_data/aryan/scifact/queries_train.tsv',
        collection='/ml_data/aryan/scifact/collections.tsv',
        config=config,
    )

    trainer.train(checkpoint='jinaai/jina-colbert-v1-en')

if __name__ == '__main__':
  main()


#> Starting...


/home/aryan/ColBERT/.venv/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


nranks = 1 	 num_gpus = 1 	 device=0
{
    "query_token_id": "[unused0]",
    "doc_token_id": "[unused1]",
    "query_token": "[Q]",
    "doc_token": "[D]",
    "ncells": null,
    "centroid_score_threshold": null,
    "ndocs": null,
    "load_index_with_mmap": false,
    "index_path": null,
    "index_bsize": 64,
    "nbits": 1,
    "kmeans_niters": 4,
    "resume": false,
    "pool_factor": 1,
    "clustering_mode": "hierarchical",
    "protected_tokens": 0,
    "similarity": "cosine",
    "bsize": 16,
    "accumsteps": 1,
    "lr": 7e-6,
    "maxsteps": 290,
    "save_every": null,
    "warmup": 30,
    "warmup_bert": null,
    "relu": false,
    "nway": 2,
    "use_ib_negatives": true,
    "reranker": false,
    "distillation_alpha": 1.0,
    "ignore_scores": false,
    "model_name": null,
    "query_maxlen": 32,
    "attend_to_mask_tokens": false,
    "interaction": "colbert",
    "dim": 128,
    "doc_maxlen": 220,
    "mask_punctuation": true,
    "checkpoint": "jinaai\/jina-colb

/home/aryan/ColBERT/.venv/lib/python3.12/site-packages/colbert/utils/amp.py:12: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler()
/home/aryan/ColBERT/.venv/lib/python3.12/site-packages/colbert/utils/amp.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  return torch.cuda.amp.autocast() if self.activated else NullContextManager()
/home/aryan/ColBERT/.venv/lib/python3.12/site-packages/colbert/utils/amp.py:35: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-ra


#> LR will use 30 warmup steps and linear decay over 290 steps.

#> QueryTokenizer.tensorize(batch_text[0], batch_background[0], bsize) ==
#> Input: 0-dimensional biomaterials lack inductive properties., 		 True, 		 None
#> Output IDs: torch.Size([32]), tensor([  101,     1,  1014,  1011,  8789, 16012,  8585, 14482,  2015,  3768,
        27427, 14194,  6024,  5144,  1012,   102,   103,   103,   103,   103,
          103,   103,   103,   103,   103,   103,   103,   103,   103,   103,
          103,   103], device='cuda:0')
#> Output Mask: torch.Size([32]), tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0], device='cuda:0')

				 1.6206409931182861 1.4870051145553589
#>>>    16.76 16.64 		|		 0.120000000000001
[Oct 03, 05:56:34] 0 3.1076459884643555
				 0.7827466726303101 1.3288464546203613
#>>>    19.51 17.41 		|		 2.1000000000000014
[Oct 03, 05:56:34] 1 3.106649935722351
				 1.8975439071655273 2.170668125152588
#>>>    18

In [16]:
from colbert import Indexer, Searcher
from colbert.data import Queries
from colbert.infra import Run, RunConfig, ColBERTConfig

checkpoint_paths = list(Path('experiments/scifact-finetune-longer').glob('**/checkpoints/colbert'))
assert checkpoint_paths, 'Run the Longer Training cell first.'

CHECKPOINT_PATH = max(checkpoint_paths, key=lambda checkpoint: checkpoint.stat().st_mtime)

assert CHECKPOINT_PATH.exists()

In [17]:
with Run().context(RunConfig(nranks=1, experiment='scifact-finetune-longer-eval')):
    index_config = ColBERTConfig(nbits=2, root='/ml_data/aryan')
    indexer = Indexer(checkpoint=str(CHECKPOINT_PATH), config=index_config)
    indexer.index(
        name='scifact.longer.finetuned.nbits=2',
        collection='/ml_data/aryan/scifact/collections.tsv',
    )




[Oct 03, 05:58:07] #> Creating directory /home/aryan/ColBERT/finetuning/experiments/scifact-finetune-longer-eval/indexes/scifact.longer.finetuned.nbits=2 


#> Starting...


/home/aryan/ColBERT/.venv/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


nranks = 1 	 num_gpus = 1 	 device=0
{
    "query_token_id": "[unused0]",
    "doc_token_id": "[unused1]",
    "query_token": "[Q]",
    "doc_token": "[D]",
    "ncells": null,
    "centroid_score_threshold": null,
    "ndocs": null,
    "load_index_with_mmap": false,
    "index_path": null,
    "index_bsize": 64,
    "nbits": 2,
    "kmeans_niters": 4,
    "resume": false,
    "pool_factor": 1,
    "clustering_mode": "hierarchical",
    "protected_tokens": 0,
    "similarity": "cosine",
    "bsize": 64,
    "accumsteps": 1,
    "lr": 7e-6,
    "maxsteps": 290,
    "save_every": null,
    "warmup": 30,
    "warmup_bert": null,
    "relu": false,
    "nway": 2,
    "use_ib_negatives": true,
    "reranker": false,
    "distillation_alpha": 1.0,
    "ignore_scores": false,
    "model_name": null,
    "query_maxlen": 32,
    "attend_to_mask_tokens": false,
    "interaction": "colbert",
    "dim": 128,
    "doc_maxlen": 160,
    "mask_punctuation": true,
    "checkpoint": "experiments\/scif

/home/aryan/ColBERT/.venv/lib/python3.12/site-packages/colbert/utils/amp.py:12: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler()



[Oct 03, 05:58:14] [0] 		 # of sampled PIDs = 5183 	 sampled_pids[:3] = [3412, 83, 2446]
[Oct 03, 05:58:14] [0] 		 #> Encoding 5183 passages..


...

[Oct 03, 05:58:17] [0] 		 avg_doclen_est = 141.1859893798828 	 len(local_sample) = 5,183
[Oct 03, 05:58:17] [0] 		 Creating 8,192 partitions.
[Oct 03, 05:58:17] [0] 		 *Estimated* 731,766 embeddings.
[Oct 03, 05:58:17] [0] 		 #> Saving the indexing plan to /home/aryan/ColBERT/finetuning/experiments/scifact-finetune-longer-eval/indexes/scifact.longer.finetuned.nbits=2/plan.json ..
Clustering 695179 points in 128D to 8192 clusters, redo 1 times, 4 iterations
  Preprocessing in 0.05 s
[Oct 03, 05:58:18] Loading decompress_residuals_cpp extension (set COLBERT_LOAD_TORCH_EXTENSION_VERBOSE=True for more info)...
[Oct 03, 05:58:18] Loading packbits_cpp extension (set COLBERT_LOAD_TORCH_EXTENSION_VERBOSE=True for more info)...
[0.035, 0.033, 0.035, 0.037, 0.037, 0.034, 0.036, 0.035, 0.034, 0.034, 0.038, 0.035, 0.037, 0.033, 0.034, 0.037, 0.036, 0.038, 0.035, 0.041, 0.035, 0.035, 0.035, 0.035, 0.037, 0.034, 0.035, 0.036, 0.034, 0.035, 0.038, 0.037, 0.035, 0.034, 0.037, 0.032, 0.034, 0.036, 0.03

0it [00:00, ?it/s]/home/aryan/ColBERT/.venv/lib/python3.12/site-packages/colbert/utils/amp.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  return torch.cuda.amp.autocast() if self.activated else NullContextManager()


[Oct 03, 05:58:21] [0] 		 #> Saving chunk 0: 	 5,183 passages and 731,767 embeddings. From #0 onward.
[Oct 03, 05:58:21] [0] 		 #> Checking all files were saved...
[Oct 03, 05:58:21] [0] 		 Found all files!
[Oct 03, 05:58:21] [0] 		 #> Building IVF...
[Oct 03, 05:58:21] [0] 		 #> Loading codes...
[Oct 03, 05:58:21] [0] 		 Sorting codes...
[Oct 03, 05:58:21] [0] 		 Getting unique codes...
[Oct 03, 05:58:21] #> Optimizing IVF to store map from centroids to list of pids..
[Oct 03, 05:58:21] #> Building the emb2pid mapping..
[Oct 03, 05:58:21] len(emb2pid) = 731767
[Oct 03, 05:58:21] #> Saved optimized IVF to /home/aryan/ColBERT/finetuning/experiments/scifact-finetune-longer-eval/indexes/scifact.longer.finetuned.nbits=2/ivf.pid.pt
[Oct 03, 05:58:21] [0] 		 #> Saving the indexing metadata to /home/aryan/ColBERT/finetuning/experiments/scifact-finetune-longer-eval/indexes/scifact.longer.finetuned.nbits=2/metadata.json ..


1it [00:02,  2.43s/it]
100%|██████████| 8192/8192 [00:00<00:00, 132674.35it/s]



#> Joined...


In [18]:
with Run().context(RunConfig(nranks=1, experiment='scifact-finetune-longer-eval')):
    search_config = ColBERTConfig(root='/ml_data/aryan')
    searcher = Searcher(
        index='scifact.longer.finetuned.nbits=2',
        checkpoint=str(CHECKPOINT_PATH),
        config=search_config,
    )
    test_queries = Queries('/ml_data/aryan/scifact/queries_test.tsv')
    finetuned_ranking = searcher.search_all(test_queries, k=100)
    finetuned_ranking_path = finetuned_ranking.save('scifact.longer.finetuned.nbits=2.ranking.tsv')

[Oct 03, 05:58:22] #> Loading collection...
0M 
[Oct 03, 05:58:25] #> Loading codec...
[Oct 03, 05:58:26] #> Loading IVF...
[Oct 03, 05:58:26] #> Loading doclens...


/home/aryan/ColBERT/.venv/lib/python3.12/site-packages/colbert/utils/amp.py:12: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler()
100%|███████████████| 1/1 [00:00<00:00, 1904.77it/s]

[Oct 03, 05:58:26] #> Loading codes and residuals...



100%|█████████████████| 1/1 [00:00<00:00, 34.70it/s]

[Oct 03, 05:58:26] #> Loading the queries from /ml_data/aryan/scifact/queries_test.tsv ...
[Oct 03, 05:58:26] #> Got 300 queries. All QIDs are unique.




/home/aryan/ColBERT/.venv/lib/python3.12/site-packages/colbert/utils/amp.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  return torch.cuda.amp.autocast() if self.activated else NullContextManager()
300it [00:01, 261.62it/s]



[Oct 03, 05:58:27] #> Creating directory /home/aryan/ColBERT/finetuning/experiments/scifact-finetune-longer-eval/none/2026-10/03/05.19.27 


[Oct 03, 05:58:27] #> Saved ranking of 300 queries and 30000 lines to /home/aryan/ColBERT/finetuning/experiments/scifact-finetune-longer-eval/none/2026-10/03/05.19.27/scifact.longer.finetuned.nbits=2.ranking.tsv


In [19]:
with open('/ml_data/aryan/scifact/i-to-ids.json', 'r', encoding='utf-8') as f:
    i_to_id = json.load(f)

with open('/ml_data/aryan/scifact/i-to-qids.json', 'r', encoding='utf-8') as f:
    i_to_qid = json.load(f)

qrels = defaultdict(list)
with open('/ml_data/aryan/scifact/qrels_test.tsv', 'r', encoding='utf-8') as f:
    for line in f:
        qid, docid = line.rstrip('\n').split('\t')
        qrels[qid].append(docid)

In [20]:
preds = defaultdict(list)
for qid, results in finetuned_ranking.items():
    actual_qid = i_to_qid[str(qid)]
    for docid, _, _ in results:
        preds[actual_qid].append(i_to_id[str(docid)])

compute_metrics(preds, qrels);

NDCG@10: 0.6700
MRR@10: 0.6311
Recall@100: 0.9097
MAP@10: 0.6184


# Even Longer Training

In [1]:
import os

os.environ['CUDA_VISIBLE_DEVICES'] = '2'

import json
from collections import defaultdict
from pathlib import Path

import ir_datasets
from colbert import Trainer
from colbert.infra import Run, RunConfig, ColBERTConfig

/home/aryan/ColBERT/.venv/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


In [2]:
def main():
  with Run().context(RunConfig(nranks=1, experiment='scifact-finetune-even-longer')):

    config = ColBERTConfig(
        bsize=16,
        lr=5e-6,
        warmup=46,
        maxsteps=464,
        nway=2,
        use_ib_negatives=True,
        root='/ml_data/aryan',
    )

    trainer = Trainer(
        triples='/ml_data/aryan/scifact/triples_train.jsonl',
        queries='/ml_data/aryan/scifact/queries_train.tsv',
        collection='/ml_data/aryan/scifact/collections.tsv',
        config=config,
    )

    trainer.train(checkpoint='jinaai/jina-colbert-v1-en')

if __name__ == '__main__':
  main()


#> Starting...


/home/aryan/ColBERT/.venv/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


nranks = 1 	 num_gpus = 1 	 device=0
{
    "query_token_id": "[unused0]",
    "doc_token_id": "[unused1]",
    "query_token": "[Q]",
    "doc_token": "[D]",
    "ncells": null,
    "centroid_score_threshold": null,
    "ndocs": null,
    "load_index_with_mmap": false,
    "index_path": null,
    "index_bsize": 64,
    "nbits": 1,
    "kmeans_niters": 4,
    "resume": false,
    "pool_factor": 1,
    "clustering_mode": "hierarchical",
    "protected_tokens": 0,
    "similarity": "cosine",
    "bsize": 16,
    "accumsteps": 1,
    "lr": 5e-6,
    "maxsteps": 464,
    "save_every": null,
    "warmup": 46,
    "warmup_bert": null,
    "relu": false,
    "nway": 2,
    "use_ib_negatives": true,
    "reranker": false,
    "distillation_alpha": 1.0,
    "ignore_scores": false,
    "model_name": null,
    "query_maxlen": 32,
    "attend_to_mask_tokens": false,
    "interaction": "colbert",
    "dim": 128,
    "doc_maxlen": 220,
    "mask_punctuation": true,
    "checkpoint": "jinaai\/jina-colb

/home/aryan/ColBERT/.venv/lib/python3.12/site-packages/colbert/utils/amp.py:12: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler()
/home/aryan/ColBERT/.venv/lib/python3.12/site-packages/colbert/utils/amp.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  return torch.cuda.amp.autocast() if self.activated else NullContextManager()
/home/aryan/ColBERT/.venv/lib/python3.12/site-packages/colbert/utils/amp.py:35: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-ra


#> LR will use 46 warmup steps and linear decay over 464 steps.

#> QueryTokenizer.tensorize(batch_text[0], batch_background[0], bsize) ==
#> Input: 0-dimensional biomaterials lack inductive properties., 		 True, 		 None
#> Output IDs: torch.Size([32]), tensor([  101,     1,  1014,  1011,  8789, 16012,  8585, 14482,  2015,  3768,
        27427, 14194,  6024,  5144,  1012,   102,   103,   103,   103,   103,
          103,   103,   103,   103,   103,   103,   103,   103,   103,   103,
          103,   103], device='cuda:0')
#> Output Mask: torch.Size([32]), tensor([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0], device='cuda:0')

				 1.6206409931182861 1.4870051145553589
#>>>    16.76 16.64 		|		 0.120000000000001
[Oct 03, 06:14:36] 0 3.1076459884643555
				 0.7827466726303101 1.3288464546203613
#>>>    19.51 17.41 		|		 2.1000000000000014
[Oct 03, 06:14:36] 1 3.106649935722351
				 1.8979829549789429 2.1708216667175293
#>>>    1

In [3]:
from colbert import Indexer, Searcher
from colbert.data import Queries
from colbert.infra import Run, RunConfig, ColBERTConfig

checkpoint_paths = list(Path('experiments/scifact-finetune-even-longer').glob('**/checkpoints/colbert'))
assert checkpoint_paths, 'Run the Even Longer Training cell first.'

CHECKPOINT_PATH = max(checkpoint_paths, key=lambda checkpoint: checkpoint.stat().st_mtime)

assert CHECKPOINT_PATH.exists()

In [4]:
with Run().context(RunConfig(nranks=1, experiment='scifact-finetune-even-longer-eval')):
    index_config = ColBERTConfig(nbits=2, root='/ml_data/aryan')
    indexer = Indexer(checkpoint=str(CHECKPOINT_PATH), config=index_config)
    indexer.index(
        name='scifact.even.longer.finetuned.nbits=2',
        collection='/ml_data/aryan/scifact/collections.tsv',
    )




[Oct 03, 06:14:43] #> Creating directory /home/aryan/ColBERT/finetuning/experiments/scifact-finetune-even-longer-eval/indexes/scifact.even.longer.finetuned.nbits=2 


#> Starting...


/home/aryan/ColBERT/.venv/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


nranks = 1 	 num_gpus = 1 	 device=0
{
    "query_token_id": "[unused0]",
    "doc_token_id": "[unused1]",
    "query_token": "[Q]",
    "doc_token": "[D]",
    "ncells": null,
    "centroid_score_threshold": null,
    "ndocs": null,
    "load_index_with_mmap": false,
    "index_path": null,
    "index_bsize": 64,
    "nbits": 2,
    "kmeans_niters": 4,
    "resume": false,
    "pool_factor": 1,
    "clustering_mode": "hierarchical",
    "protected_tokens": 0,
    "similarity": "cosine",
    "bsize": 64,
    "accumsteps": 1,
    "lr": 5e-6,
    "maxsteps": 464,
    "save_every": null,
    "warmup": 46,
    "warmup_bert": null,
    "relu": false,
    "nway": 2,
    "use_ib_negatives": true,
    "reranker": false,
    "distillation_alpha": 1.0,
    "ignore_scores": false,
    "model_name": null,
    "query_maxlen": 32,
    "attend_to_mask_tokens": false,
    "interaction": "colbert",
    "dim": 128,
    "doc_maxlen": 160,
    "mask_punctuation": true,
    "checkpoint": "experiments\/scif

/home/aryan/ColBERT/.venv/lib/python3.12/site-packages/colbert/utils/amp.py:12: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler()



[Oct 03, 06:14:50] [0] 		 # of sampled PIDs = 5183 	 sampled_pids[:3] = [3412, 83, 2446]
[Oct 03, 06:14:50] [0] 		 #> Encoding 5183 passages..


...

[Oct 03, 06:14:53] [0] 		 avg_doclen_est = 141.1859893798828 	 len(local_sample) = 5,183
[Oct 03, 06:14:53] [0] 		 Creating 8,192 partitions.
[Oct 03, 06:14:53] [0] 		 *Estimated* 731,766 embeddings.
[Oct 03, 06:14:53] [0] 		 #> Saving the indexing plan to /home/aryan/ColBERT/finetuning/experiments/scifact-finetune-even-longer-eval/indexes/scifact.even.longer.finetuned.nbits=2/plan.json ..
Clustering 695179 points in 128D to 8192 clusters, redo 1 times, 4 iterations
  Preprocessing in 0.05 s
[Oct 03, 06:14:54] Loading decompress_residuals_cpp extension (set COLBERT_LOAD_TORCH_EXTENSION_VERBOSE=True for more info)...
[Oct 03, 06:14:54] Loading packbits_cpp extension (set COLBERT_LOAD_TORCH_EXTENSION_VERBOSE=True for more info)...
[0.035, 0.034, 0.036, 0.038, 0.037, 0.035, 0.036, 0.035, 0.035, 0.035, 0.038, 0.035, 0.038, 0.034, 0.035, 0.037, 0.037, 0.038, 0.036, 0.041, 0.036, 0.035, 0.036, 0.036, 0.037, 0.034, 0.036, 0.036, 0.035, 0.036, 0.038, 0.037, 0.036, 0.034, 0.037, 0.033, 0.034, 0

0it [00:00, ?it/s]/home/aryan/ColBERT/.venv/lib/python3.12/site-packages/colbert/utils/amp.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  return torch.cuda.amp.autocast() if self.activated else NullContextManager()


[Oct 03, 06:14:57] [0] 		 #> Saving chunk 0: 	 5,183 passages and 731,767 embeddings. From #0 onward.
[Oct 03, 06:14:57] [0] 		 #> Checking all files were saved...
[Oct 03, 06:14:57] [0] 		 Found all files!
[Oct 03, 06:14:57] [0] 		 #> Building IVF...
[Oct 03, 06:14:57] [0] 		 #> Loading codes...
[Oct 03, 06:14:57] [0] 		 Sorting codes...
[Oct 03, 06:14:57] [0] 		 Getting unique codes...
[Oct 03, 06:14:57] #> Optimizing IVF to store map from centroids to list of pids..
[Oct 03, 06:14:57] #> Building the emb2pid mapping..
[Oct 03, 06:14:57] len(emb2pid) = 731767
[Oct 03, 06:14:57] #> Saved optimized IVF to /home/aryan/ColBERT/finetuning/experiments/scifact-finetune-even-longer-eval/indexes/scifact.even.longer.finetuned.nbits=2/ivf.pid.pt
[Oct 03, 06:14:57] [0] 		 #> Saving the indexing metadata to /home/aryan/ColBERT/finetuning/experiments/scifact-finetune-even-longer-eval/indexes/scifact.even.longer.finetuned.nbits=2/metadata.json ..


1it [00:02,  2.52s/it]
100%|██████████| 8192/8192 [00:00<00:00, 128884.62it/s]



#> Joined...


In [5]:
with Run().context(RunConfig(nranks=1, experiment='scifact-finetune-even-longer-eval')):
    search_config = ColBERTConfig(root='/ml_data/aryan')
    searcher = Searcher(
        index='scifact.even.longer.finetuned.nbits=2',
        checkpoint=str(CHECKPOINT_PATH),
        config=search_config,
    )
    test_queries = Queries('/ml_data/aryan/scifact/queries_test.tsv')
    finetuned_ranking = searcher.search_all(test_queries, k=100)
    finetuned_ranking_path = finetuned_ranking.save('scifact.even.longer.finetuned.nbits=2.ranking.tsv')

[Oct 03, 06:14:58] #> Loading collection...
0M 
[Oct 03, 06:15:01] #> Loading codec...
[Oct 03, 06:15:01] Loading decompress_residuals_cpp extension (set COLBERT_LOAD_TORCH_EXTENSION_VERBOSE=True for more info)...


/home/aryan/ColBERT/.venv/lib/python3.12/site-packages/colbert/utils/amp.py:12: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler()


[Oct 03, 06:15:01] Loading packbits_cpp extension (set COLBERT_LOAD_TORCH_EXTENSION_VERBOSE=True for more info)...
[Oct 03, 06:15:01] #> Loading IVF...
[Oct 03, 06:15:01] #> Loading doclens...


100%|███████████████| 1/1 [00:00<00:00, 1866.62it/s]

[Oct 03, 06:15:01] #> Loading codes and residuals...



100%|█████████████████| 1/1 [00:00<00:00, 30.37it/s]

[Oct 03, 06:15:01] #> Loading the queries from /ml_data/aryan/scifact/queries_test.tsv ...
[Oct 03, 06:15:01] #> Got 300 queries. All QIDs are unique.




/home/aryan/ColBERT/.venv/lib/python3.12/site-packages/colbert/utils/amp.py:15: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  return torch.cuda.amp.autocast() if self.activated else NullContextManager()
300it [00:01, 207.91it/s]




[Oct 03, 06:15:03] #> Creating directory /home/aryan/ColBERT/finetuning/experiments/scifact-finetune-even-longer-eval/none/2026-10/03/06.14.22 


[Oct 03, 06:15:03] #> Saved ranking of 300 queries and 30000 lines to /home/aryan/ColBERT/finetuning/experiments/scifact-finetune-even-longer-eval/none/2026-10/03/06.14.22/scifact.even.longer.finetuned.nbits=2.ranking.tsv


In [6]:
with open('/ml_data/aryan/scifact/i-to-ids.json', 'r', encoding='utf-8') as f:
    i_to_id = json.load(f)

with open('/ml_data/aryan/scifact/i-to-qids.json', 'r', encoding='utf-8') as f:
    i_to_qid = json.load(f)

qrels = defaultdict(list)
with open('/ml_data/aryan/scifact/qrels_test.tsv', 'r', encoding='utf-8') as f:
    for line in f:
        qid, docid = line.rstrip('\n').split('\t')
        qrels[qid].append(docid)

In [10]:
preds = defaultdict(list)
for qid, results in finetuned_ranking.items():
    actual_qid = i_to_qid[str(qid)]
    for docid, _, _ in results:
        preds[actual_qid].append(i_to_id[str(docid)])

compute_metrics(preds, qrels);

NDCG@10: 0.6638
MRR@10: 0.6243
Recall@100: 0.9063
MAP@10: 0.6093
